In [1]:
# [CSV 경로 문자열 영구 보정 스크립트]

import pandas as pd
from pathlib import Path

# 현재 워크스페이스 및 파일 설정
BASE_DIR = Path(r"C:\Users\SSAFY\Downloads\2026-ssafy-ai-15-2")
CSV_PATH = BASE_DIR / "train_final.csv"

def fix_csv_paths():
    print("CSV 파일 경로 보정 시작...")
    df = pd.read_csv(CSV_PATH)
    
    # 기존에 잘못 기록된 path 컬럼을 무시하고, id(파일명)를 기준으로 올바른 상대 경로 재할당
    # 현재 실제 구조: 2026-ssafy-ai-15-2 / augmented_images / images / 파일명
    df['path'] = df['id'].apply(lambda filename: f"augmented_images/images/{filename}")
    
    # 덮어쓰기 저장 (UTF-8-SIG 인코딩 유지)
    df.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
    
    # 검증 출력
    print("\n✅ 경로 보정 완료! 업데이트된 샘플:")
    for idx, row in df.head(3).iterrows():
        print(f" - ID: {row['id']} \n   -> PATH: {row['path']}")

# 실행
fix_csv_paths()

CSV 파일 경로 보정 시작...

✅ 경로 보정 완료! 업데이트된 샘플:
 - ID: 플라스틱류-밀폐용기-밀폐용기_24_X011_C196_1207_0_aug2_color_strong.jpg 
   -> PATH: augmented_images/images/플라스틱류-밀폐용기-밀폐용기_24_X011_C196_1207_0_aug2_color_strong.jpg
 - ID: 플라스틱류-장남감-장남감_24_X339_C509_0329_0_aug2_color.jpg 
   -> PATH: augmented_images/images/플라스틱류-장남감-장남감_24_X339_C509_0329_0_aug2_color.jpg
 - ID: 비닐-포장제-포장제_15_X418_C889_0330_1_original.jpg 
   -> PATH: augmented_images/images/비닐-포장제-포장제_15_X418_C889_0330_1_original.jpg


In [4]:
%pip install --upgrade transformers accelerate bitsandbytes peft

   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
    --------------------------------------- 0.1/10.2 MB 2.8 MB/s eta 0:00:04
   ------ --------------------------------- 1.7/10.2 MB 17.9 MB/s eta 0:00:01
   ------------------------ --------------- 6.4/10.2 MB 50.8 MB/s eta 0:00:01
   ---------------------------------------  10.2/10.2 MB 59.5 MB/s eta 0:00:01
   ---------------------------------------- 10.2/10.2 MB 54.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.4.0
    Uninstalling transformers-5.4.0:
      Successfully uninstalled transformers-5.4.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.3.18 requires torch<2.11.0,>=2.4.0, but you have torch 2.12.0.dev20260401+cu128 which is incompatible.
unsloth 2026.3.18 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.3.0,>=4.51.3, but you have transformers 5.5.0 which is incompatible.
unsloth-zoo 2026.3.7 requires torch<2.11.0,>=2.4.0, but you have torch 2.12.0.dev20260401+cu128 which is incompatible.
unsloth-zoo 2026.3.7 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.3.0,>=4.51.3, but you have transformers 5.5.0 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# [VLM 전용: DataCollator 기반 동적 패딩 & QLoRA 학습 파이프라인]

import os
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from transformers import (
    AutoProcessor, 
    Qwen2VLForConditionalGeneration, 
    BitsAndBytesConfig, 
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ──────────────────────────────────────────────
# 1. 환경 및 경로 설정
# ──────────────────────────────────────────────
BASE_DIR = Path(r"C:\Users\SSAFY\Downloads\2026-ssafy-ai-15-2")
TRAIN_CSV = BASE_DIR / "train_final.csv"
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct" 
OUTPUT_DIR = BASE_DIR / "vqa_lora_model"

# ──────────────────────────────────────────────
# 2. VQA 전용 Dataset (순수 데이터만 전달, 텐서화는 위임)
# ──────────────────────────────────────────────
class QwenVQADataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # [핵심 방어코드] HF Trainer가 list나 slice로 검증을 시도할 때의 충돌 방지
        if isinstance(idx, (list, slice)):
            return [self.__getitem__(i) for i in (range(len(self))[idx] if isinstance(idx, slice) else idx)]
        
        row = self.df.iloc[idx]
        image_path = str(BASE_DIR / row['path'])
        
        prompt = (
            "You are a helpful AI assistant for visual question answering. "
            "Analyze the image and choose the correct answer among options 'a', 'b', 'c', or 'd'. "
            "Output only the alphabet of the correct answer.\n"
            f"Question: {row['question']}\nOptions:\na) {row['a']}\nb) {row['b']}\nc) {row['c']}\nd) {row['d']}\nAnswer:"
        )
        
        messages = [
            {"role": "user", "content": [{"type": "image", "image": image_path}, {"type": "text", "text": prompt}]},
            {"role": "assistant", "content": [{"type": "text", "text": str(row['answer'])}]}
        ]
        
        return {"messages": messages, "image_path": image_path}

# ──────────────────────────────────────────────
# 3. VLM 전용 DataCollator (동적 배치 패딩 처리)
# ──────────────────────────────────────────────
# [수정된 VQADataCollator: 리스트 호출 에러 방지 및 텐서 정규화]

# [최종 수정본: Processor 리스트 에러 및 텐서 배치 에러 완벽 방어]

class VQADataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        # 1. 검증 단계에서 리스트로 들어오는 경우 방어
        if isinstance(features, dict):
            features = [features]
            
        texts = []
        images = []
        
        for feature in features:
            if isinstance(feature, list): feature = feature[0]
            
            # 메시지 템플릿 적용
            messages = feature["messages"]
            text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            texts.append(text)
            
            # 이미지 로드
            img = Image.open(feature["image_path"]).convert("RGB")
            images.append(img)
            
        # 2. [수정 포인트] 프로세서 직접 호출 대신 내부 메서드 활용 가능성 고려
        # 가장 안전한 방식은 인자를 명확히 전달하는 것입니다.
        try:
            inputs = self.processor(
                text=texts,
                images=images,
                padding=True,
                return_tensors="pt"
            )
        except TypeError:
            # 만약 위 방식도 안 된다면 image_processor와 tokenizer를 분리 호출
            pixel_values = self.processor.image_processor(images, return_tensors="pt")["pixel_values"]
            input_ids = self.processor.tokenizer(texts, padding=True, return_tensors="pt")["input_ids"]
            attention_mask = self.processor.tokenizer(texts, padding=True, return_tensors="pt")["attention_mask"]
            inputs = {
                "pixel_values": pixel_values,
                "input_ids": input_ids,
                "attention_mask": attention_mask
            }
        
        # 3. 레이블 생성 (패딩은 -100 처리하여 학습 제외)
        labels = inputs["input_ids"].clone()
        pad_token_id = self.processor.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100
        
        inputs["labels"] = labels
        return inputs

# ──────────────────────────────────────────────
# 학습 실행 부분 (train_pipeline 내부는 동일하게 유지)
# ──────────────────────────────────────────────
# ──────────────────────────────────────────────
# 4. 모델 로드 및 학습 로직
# ──────────────────────────────────────────────
def train_pipeline():
    print("[1/3] 데이터 로드 중...")
    df = pd.read_csv(TRAIN_CSV).dropna(subset=['question', 'answer']) 
    
    train_df = df.sample(frac=0.95, random_state=42)
    eval_df = df.drop(train_df.index)
    
    # Dataset 초기화 (Processor 제거)
    train_dataset = QwenVQADataset(train_df)
    eval_dataset = QwenVQADataset(eval_df)
    
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    data_collator = VQADataCollator(processor)
    
    print("[2/3] 모델 양자화(4-bit) 및 LoRA 어댑터 결합 중...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto"
    )
    model = prepare_model_for_kbit_training(model)
    
    lora_config = LoraConfig(
        r=16, lora_alpha=32, target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], 
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    
    print("[3/3] 파인튜닝 루프 진입...")
    training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    optim="paged_adamw_32bit",
    save_steps=500,                # 너무 자주 저장하면 느려지므로 상향
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    max_steps=1000,                # 학습 진행을 위해 스텝 수 확보
    eval_strategy="steps",         # [수정] evaluation_strategy -> eval_strategy
    eval_steps=500,                # 검증 빈도 조절
    remove_unused_columns=False,
    logging_first_step=True,
    report_to="none",              # WandB 등 외부 로깅 충돌 방지
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,  # [핵심] 커스텀 DataCollator 주입
    )
    
    trainer.train()
    trainer.model.save_pretrained(OUTPUT_DIR / "final_lora")
    print("✅ 모델 학습 및 저장 완료!")

if __name__ == "__main__":
    torch.cuda.empty_cache()
    train_pipeline()

[1/3] 데이터 로드 중...
[2/3] 모델 양자화(4-bit) 및 LoRA 어댑터 결합 중...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[3/3] 파인튜닝 루프 진입...


c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\eval_frame.py:1272: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
500,2.554430,2.541692
1000,2.535180,2.533993


c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\eval_frame.py:1272: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


✅ 모델 학습 및 저장 완료!


In [8]:
# [최종 추론 및 제출 파일 생성 파이프라인]

import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel

# 1. 경로 설정
BASE_DIR = Path(r"C:\Users\SSAFY\Downloads\2026-ssafy-ai-15-2")
TEST_CSV = BASE_DIR / "test.csv"
LORA_PATH = BASE_DIR / "vqa_lora_model" / "final_lora"
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
OUTPUT_CSV = BASE_DIR / "submission_final.csv"

def run_inference():
    print("[1/3] 학습된 LoRA 모델 로드 중...")
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    
    # 베이스 모델 로드 (4-bit)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto"
    )
    # 학습한 어댑터(LoRA) 병합
    model = PeftModel.from_pretrained(model, LORA_PATH)
    model.eval()

    print("[2/3] 테스트 데이터 추론 시작 (약 5,000장)...")
    test_df = pd.read_csv(TEST_CSV)
    results = []

    with torch.no_grad():
        for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
            img_path = BASE_DIR / row['path']
            image = Image.open(img_path).convert("RGB")
            
            # 질문 구성 (학습 때와 동일한 포맷 필수)
            prompt = (
                "Analyze the image and choose the correct answer among options 'a', 'b', 'c', or 'd'. "
                "Output only the alphabet of the correct answer.\n"
                f"Question: {row['question']}\nOptions:\na) {row['a']}\nb) {row['b']}\nc) {row['c']}\nd) {row['d']}\nAnswer:"
            )
            
            messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda")

            # 답변 생성
            output_ids = model.generate(**inputs, max_new_tokens=5)
            generated_text = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
            
            # 'ASSISTANT: b' 형태에서 마지막 알파벳만 추출하는 방어 로직
            final_ans = generated_text.split("Answer:")[-1].strip().lower()
            # a, b, c, d 중 하나가 아니면 첫 번째 글자만 따오기
            final_ans = final_ans[0] if final_ans and final_ans[0] in ['a', 'b', 'c', 'd'] else 'a'
            
            results.append({"id": row['id'], "answer": final_ans})

    print(f"[3/3] 결과 저장 중: {OUTPUT_CSV}")
    submission_df = pd.DataFrame(results)
    submission_df.to_csv(OUTPUT_CSV, index=False)
    print("✅ 제출 파일 생성 완료! 캐글에 업로드하세요.")

if __name__ == "__main__":
    run_inference()

[1/3] 학습된 LoRA 모델 로드 중...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

[2/3] 테스트 데이터 추론 시작 (약 5,000장)...


100%|██████████| 5074/5074 [49:13<00:00,  1.72it/s] 

[3/3] 결과 저장 중: C:\Users\SSAFY\Downloads\2026-ssafy-ai-15-2\submission_final.csv
✅ 제출 파일 생성 완료! 캐글에 업로드하세요.
